# SDOH / neighborhood / environmental exposure -- concept discovery

Searches the real CDR for candidate concept_ids covering the phenotype list below,
rather than guessing -- this repo's own history is full of plausible-looking
concept_ids that turned out wrong when actually checked (`hip_circumference`'s LOINC
mapped to *thigh* circumference; the PPI alcohol-frequency items were unpopulated
under their own vocabulary's concept_ids and had to be traced to what they actually
"Maps to"). Same concept-then-count pattern `01_query_filter_check.ipynb`'s
appendices already use: search `{CDR}.concept` by name pattern, then count real rows
per candidate against `{CDR}.observation`, so a plausible name with zero real data
doesn't get mistaken for a usable phenotype.

| Domain | Phenotype |
|---|---|
| Individual SES | years of education, household income, employment, insurance |
| Financial | food insecurity, financial strain |
| Social | loneliness, social support, perceived stress |
| Neighborhood | deprivation index, median income, poverty, education level, uninsured fraction |
| Environment | PM2.5, NO2, ozone, NDVI, noise exposure |

**Individual SES/Financial/Social** are almost certainly AoU's "Basics" survey
(income/employment/education/insurance) and "Social Determinants of Health" (SDOH)
survey module (food insecurity, financial strain, loneliness -- likely the UCLA-3
scale, perceived stress -- likely PSS) -- both real PPI-vocabulary survey
instruments, but their exact concept_ids need confirming here, not assuming.

**Neighborhood** overlaps with `zip3_ses_map`, which `residualize_phenotypes.ipynb`'s
`pull_covariates()` already reads for `median_income`/`poverty`/`deprivation_index`
-- checks that table's full column list below for `education level`/
`uninsured fraction` before assuming they need a separate source.

**Environment** (PM2.5/NO2/ozone/NDVI/noise) -- not assumed to exist in this CDR
version at all; searches `INFORMATION_SCHEMA.TABLES` directly rather than guessing
a table name.

All output is aggregate concept metadata + counts, never person-level, same
convention as every other appendix in this repo.

## Compute resource

Small aggregate BigQuery queries only -- Workbench 2.0's default (2 CPU / 13 GB) is enough.

In [ ]:
required_pkgs <- c("dplyr", "readr", "stringr", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)

con <- aou_connect()
run_query <- function(sql) collect(aou_sql(sql))

## Individual SES / Financial / Social: search PPI survey concepts

Searches `{CDR}.concept` (vocabulary_id = `'PPI'`, the AoU survey vocabulary) for
each phenotype's keyword pattern(s), then counts real `{CDR}.observation` rows per
candidate concept_id -- same pattern as `01_query_filter_check.ipynb`'s lifestyle
appendix. `concept_code` is included since AoU's PPI codes (e.g. `Employment_*`,
`Insurance_*`, `SDOH_*`) are often a clearer signal than the free-text
`concept_name` alone for telling which survey module an item belongs to.

Keyword patterns below are a starting point, not confirmed -- inspect the results
and narrow/broaden them if a phenotype's real item doesn't show up, same as this
repo's existing lifestyle/waist-hip appendices needed real iteration to land on
the right concept_ids.

In [ ]:
PPI_SEARCH_TERMS <- list(
  years_of_education    = c("%education%", "%grade%school%"),
  household_income       = c("%income%"),
  employment              = c("%employ%"),
  insurance                = c("%insurance%"),
  food_insecurity          = c("%food%worr%", "%food%afford%", "%food%insecur%", "%hungry%"),
  financial_strain         = c("%financial%", "%money%", "%unable to pay%"),
  loneliness                = c("%lonel%", "%isolat%"),
  social_support            = c("%social support%", "%emotional support%", "%someone%support%"),
  perceived_stress          = c("%perceived stress%", "%stress%")
)

search_ppi_concept <- function(phenotype_name, patterns) {
  like_clauses <- paste(sprintf("LOWER(concept_name) LIKE '%s'", tolower(patterns)), collapse = " OR ")
  concepts <- run_query(sprintf("
    SELECT concept_id, concept_name, concept_code, domain_id, vocabulary_id, standard_concept
    FROM {CDR}.concept
    WHERE vocabulary_id = 'PPI'
      AND (%s)
  ", like_clauses))
  if (nrow(concepts) == 0) {
    return(tibble(phenotype_name, concept_id = NA_integer_, concept_name = NA_character_,
                   concept_code = NA_character_, n_persons = NA_integer_, n_rows = NA_integer_))
  }

  counts <- bind_rows(lapply(concepts$concept_id, function(cid) {
    run_query(sprintf("
      SELECT COUNT(DISTINCT person_id) AS n_persons, COUNT(*) AS n_rows
      FROM {CDR}.observation
      WHERE observation_concept_id = %s
    ", cid)) %>% mutate(concept_id = cid, .before = 1)
  }))

  concepts %>%
    inner_join(counts, by = "concept_id") %>%
    mutate(phenotype_name, .before = 1) %>%
    arrange(desc(n_persons))
}

ppi_results <- bind_rows(lapply(names(PPI_SEARCH_TERMS), function(name) {
  search_ppi_concept(name, PPI_SEARCH_TERMS[[name]])
}))

ppi_results %>% arrange(phenotype_name, desc(n_persons))

## Neighborhood: check `zip3_ses_map`'s full schema

`residualize_phenotypes.ipynb`'s `pull_covariates()` already reads `median_income`/
`fraction_poverty`/`deprivation_index` from this table -- checks whether
`education level`/`uninsured fraction` are already columns here (no new join
needed) before assuming they need a separate source.

In [ ]:
zip3_columns <- run_query("
  SELECT column_name, data_type
  FROM {CDR}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_name = 'zip3_ses_map'
  ORDER BY ordinal_position
")
zip3_columns

## Environment: search for any linked exposure table

Not assumed to exist -- searches `{CDR}.INFORMATION_SCHEMA.TABLES` for any table
whose name suggests environmental/exposure data (PM2.5, NO2, ozone, NDVI, noise,
"environment", "exposure", "air quality"), rather than guessing a table name that
might not exist in this CDR version at all. If nothing turns up, that's a real,
useful negative result -- these variables would need an external
geocoded/zip-linked dataset joined in outside the CDR, not something missed by a
narrower search.

In [ ]:
ENV_TABLE_PATTERNS <- c("%environ%", "%exposure%", "%air_qual%", "%pm25%", "%pm2_5%",
                        "%ozone%", "%no2%", "%ndvi%", "%noise%", "%greenspace%")

like_clauses <- paste(sprintf("LOWER(table_name) LIKE '%s'", ENV_TABLE_PATTERNS), collapse = " OR ")
env_tables <- run_query(sprintf("
  SELECT table_name
  FROM {CDR}.INFORMATION_SCHEMA.TABLES
  WHERE (%s)
", like_clauses))

if (nrow(env_tables) == 0) {
  message("No table names matching environmental-exposure patterns found in this CDR's dataset. ",
          "This likely means PM2.5/NO2/ozone/NDVI/noise data isn't linked in this CDR version -- ",
          "would need an external geocoded dataset (e.g. EPA AQS, CDC/ATSDR environmental justice ",
          "index) joined on zip3/geography outside the CDR, not something this search missed.")
} else {
  message(sprintf("%d candidate table(s) found -- inspect their schemas next", nrow(env_tables)))
}
env_tables

In [ ]:
# Run only if sdoh-env-search found candidates above -- introspects each
# candidate table's columns so you can tell what geography/pollutant/units it
# actually carries before trying to join it to anything
if (nrow(env_tables) > 0) {
  env_table_columns <- bind_rows(lapply(env_tables$table_name, function(tbl) {
    run_query(sprintf("
      SELECT column_name, data_type
      FROM {CDR}.INFORMATION_SCHEMA.COLUMNS
      WHERE table_name = '%s'
      ORDER BY ordinal_position
    ", tbl)) %>% mutate(table_name = tbl, .before = 1)
  }))
  env_table_columns
}

## Summary

`ppi_results` -- for each phenotype, the real candidate concept_id(s) with actual
`n_persons`/`n_rows`, sorted by coverage. Confirm the top candidate is really what
it claims to be (check `concept_code`/`concept_name` against the AoU Data Browser's
"Survey" domain for that item), then add a row to `docs/phenotype_list.tsv` with
its confirmed `concept_id`, following the existing schema (`source = "survey"` for
a single-item numeric answer, `source = "survey_composite"` if a phenotype needs
summing multiple items the way `alcohol_audit_c_score` does).

`zip3_columns`/`env_table_columns` -- whatever's already available for the
Neighborhood/Environment domains without a new external join. Anything not found
here is a real gap, not a search miss -- worth flagging to the team rather than
assuming it's in the CDR somewhere.